# 모듈 6. Memory PII 보호

이 노트북은 LangGraph `MemorySaver`에 저장되는 상태에서 개인정보가 어떻게 남을 수 있는지 확인하고, 저장 전 마스킹 노드를 추가합니다.

## 학습 목표

- Checkpointer/Memory가 안전 저장소를 의미하지 않는다는 점을 이해한다.
- LangGraph 상태에 PII가 들어가기 전에 마스킹한다.
- 저장 요청과 삭제 요청을 그래프 분기로 처리한다.
- 보안 없는 A안과 보안 적용 B안을 비교한다.

## 왜 LangGraph 구조로 배워야 할까?

LangGraph의 checkpointer는 그래프 상태를 저장합니다. 즉, 상태에 전화번호나 주민등록번호가 들어가면 그대로 저장될 수 있습니다.

Memory 보안의 핵심은 "나중에 지우기"보다 **상태에 들어가기 전에 줄이기**입니다.

In [ ]:
import re
from typing import TypedDict

from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph


user_message = (
    "안녕하세요. 제 이름은 김민수이고 전화번호는 010-1234-5678입니다. "
    "주민등록번호는 900101-1234567이고 이메일은 minsu@example.com입니다."
)


class MemoryState(TypedDict, total=False):
    user_id: str
    message: str
    memory_record: str
    pii_masked: bool
    response: str

## A안. 원문을 그대로 Graph 상태에 저장하기

아래 그래프는 사용자의 메시지를 그대로 `memory_record`에 복사합니다. checkpointer는 이 상태를 저장하므로 민감정보가 남습니다.

In [ ]:
def raw_save_node(state: MemoryState) -> MemoryState:
    return {
        "memory_record": state["message"],
        "pii_masked": False,
        "response": "대화 내용을 기억했습니다.",
    }


raw_builder = StateGraph(MemoryState)
raw_builder.add_node("raw_save", raw_save_node)
raw_builder.add_edge(START, "raw_save")
raw_builder.add_edge("raw_save", END)

raw_graph = raw_builder.compile(checkpointer=MemorySaver())
raw_config = {"configurable": {"thread_id": "raw-memory-demo"}}

raw_graph.invoke(
    {"user_id": "user-123", "message": user_message},
    config=raw_config,
)

raw_snapshot = raw_graph.get_state(raw_config)
print(raw_snapshot.values["memory_record"])

## B안. 저장 전 PII 마스킹 노드 추가하기

보호 버전은 `protected_save_node`에서 전화번호, 주민등록번호, 이메일을 마스킹한 뒤 상태에 저장합니다.

In [ ]:
PII_REGEX = {
    "rrn": re.compile(r"(?<!\d)\d{6}-\d{7}(?!\d)"),
    "phone": re.compile(r"(?<!\d)01[016789]-\d{3,4}-\d{4}(?!\d)"),
    "email": re.compile(
        r"(?<![A-Za-z0-9._%+-])[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}(?![A-Za-z0-9.-])"
    ),
}


def mask_pii(text: str) -> str:
    masked = PII_REGEX["rrn"].sub("[주민등록번호 마스킹]", text)
    masked = PII_REGEX["phone"].sub("[전화번호 마스킹]", masked)
    masked = PII_REGEX["email"].sub("[이메일 마스킹]", masked)
    return masked


def protected_save_node(state: MemoryState) -> MemoryState:
    masked_message = mask_pii(state["message"])
    return {
        "memory_record": masked_message,
        "pii_masked": masked_message != state["message"],
        "response": "민감정보를 줄인 뒤 대화 내용을 기억했습니다.",
    }


protected_builder = StateGraph(MemoryState)
protected_builder.add_node("protected_save", protected_save_node)
protected_builder.add_edge(START, "protected_save")
protected_builder.add_edge("protected_save", END)

protected_graph = protected_builder.compile(checkpointer=MemorySaver())
protected_config = {"configurable": {"thread_id": "protected-memory-demo"}}

protected_graph.invoke(
    {"user_id": "user-123", "message": user_message},
    config=protected_config,
)

protected_snapshot = protected_graph.get_state(protected_config)
print(protected_snapshot.values["memory_record"])
print("pii_masked =", protected_snapshot.values["pii_masked"])

## 삭제 요청 분기 추가하기

실제 서비스에서는 사용자의 삭제 요청도 처리해야 합니다. 아래 그래프는 메시지가 삭제 요청이면 저장하지 않고 삭제 처리 응답으로 분기합니다.

In [ ]:
class MemoryRequestState(TypedDict, total=False):
    user_id: str
    message: str
    memory_record: str
    response: str
    request_type: str


def classify_memory_request(state: MemoryRequestState) -> MemoryRequestState:
    if "삭제" in state["message"] or "잊어" in state["message"]:
        return {"request_type": "forget"}
    return {"request_type": "save"}


def route_memory_request(state: MemoryRequestState) -> str:
    return state["request_type"]


def save_masked_memory(state: MemoryRequestState) -> MemoryRequestState:
    return {
        "memory_record": mask_pii(state["message"]),
        "response": "마스킹 후 저장했습니다.",
    }


def forget_memory(state: MemoryRequestState) -> MemoryRequestState:
    return {
        "memory_record": "",
        "response": f"{state['user_id']}의 개인화 메모리를 삭제 대상으로 표시했습니다.",
    }


request_builder = StateGraph(MemoryRequestState)
request_builder.add_node("classify", classify_memory_request)
request_builder.add_node("save", save_masked_memory)
request_builder.add_node("forget", forget_memory)
request_builder.add_edge(START, "classify")
request_builder.add_conditional_edges(
    "classify",
    route_memory_request,
    {"save": "save", "forget": "forget"},
)
request_builder.add_edge("save", END)
request_builder.add_edge("forget", END)

memory_request_graph = request_builder.compile()

print(
    memory_request_graph.invoke(
        {"user_id": "user-123", "message": "이전 상담 기록을 삭제해줘."}
    )["response"]
)

## 정리

- LangGraph checkpointer는 상태를 저장하므로 상태 설계 자체가 보안 설계입니다.
- PII는 `memory_record`에 들어가기 전에 마스킹해야 합니다.
- 삭제 요청은 일반 대화와 다른 흐름으로 분리해야 감사와 운영 처리가 쉬워집니다.

## 실습 과제

1. 계좌번호 패턴을 `PII_REGEX`에 추가해보세요.
2. `forget_memory`가 감사 로그 이벤트를 함께 반환하도록 상태를 확장해보세요.